# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Publisher: {getattr(metadata, 'publisher', 'N/A')}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use the `@id` specified in the Croissant schema.

In [ ]:
from pprint import pprint

# List all record sets by @id and their fields
print('Available Record Sets:')
record_sets_list = []
for record_set in dataset.record_sets():
    print(f"  Record Set: @id={record_set['@id']}  name={record_set.get('name')}")
    record_sets_list.append(record_set['@id'])

    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print('    Fields:')
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', 'No @id')
            field_name = field.get('name', None)
        else:
            field_id = field
            field_name = None
        print(f"      Field @id={field_id}{' name='+field_name if field_name else ''}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For this dataset, based on schema inspection, the main table is proteins identified from EVs.
# We will use its @id if present. Replace below with the actual @id from printed output above.
# If only one record set is present, use it. Otherwise, insert the main one.
if not record_sets_list:
    raise RuntimeError("No record sets available in the dataset schema.")

# Use the first record set for demonstration.
main_record_set_id = record_sets_list[0]
print(f'Using record set @id: {main_record_set_id}')

# Extract data from all record sets into pandas DataFrames
dataframes = {}
for rs_id in record_sets_list:
    print(f'Reading records from record set {rs_id}')
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  DataFrame shape: {df.shape}")

print('Columns in main DataFrame:')
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming numeric data, and grouping by key attributes to prepare for further analysis.

In [ ]:
# Specify a numeric field for value-based exploration.
# For example, 'cr:coveragePercentage' might be a candidate; adjust as needed based on the printed columns above.
df = dataframes[main_record_set_id]
numeric_field_id = None
for col in df.columns:
    if 'coverage' in col.lower() or 'abund' in col.lower() or 'cr:' in col or 'peptide' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: pick the first numeric-looking column
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            numeric_field_id = col
            break
if numeric_field_id is None:
    raise ValueError('No numeric field found for EDA!')
print(f'Using numeric field @id: {numeric_field_id}')

# Remove missing data
sub_df = df[[numeric_field_id]].dropna()

# Ensure numeric dtype (some CSVs may have loaded as str)
sub_df[numeric_field_id] = pd.to_numeric(sub_df[numeric_field_id], errors='coerce')
sub_df = sub_df.dropna()
# Filter records with high values in the numeric field
threshold = sub_df[numeric_field_id].quantile(0.75)  # Top quartile as threshold
filtered_df = df[df[numeric_field_id].astype(float) > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}")
print(filtered_df[[numeric_field_id]].head())

# Normalize numeric field (z-score normalization)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
filtered_df[numeric_field_id + '_normalized'] = scaler.fit_transform(filtered_df[[numeric_field_id]].astype(float))
print('Normalized column head:')
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by a string/categorical field if available (e.g., 'cr:proteinDescription' or 'cr:sample' or User can pick)
candidate_group_fields = [col for col in df.columns if (df[col].dtype=='object' and col!=numeric_field_id)]
group_field_id = candidate_group_fields[0] if candidate_group_fields else None
if group_field_id:
    print(f'Grouping by field @id: {group_field_id}')
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    )
    print(grouped_df.head())
else:
    print('No suitable group/categorical field found.')

## 5. Visualization
Visualize distributions or relationships in the main record set field.
Below, we show a histogram of the selected numeric field, and if grouping is present, visualize group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id} in Record Set {main_record_set_id}')
plt.show()

# If we have a grouping, plot top N groups by mean
if group_field_id:
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(y=group_field_id, x=numeric_field_id, data=top_groups)
    plt.title(f'Mean {numeric_field_id} by {group_field_id} (top 10)')
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and process the dataset `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` using its Croissant schema and the `mlcroissant` Python library.

We loaded the dataset, inspected available record sets and their fields using their `@id`, extracted the main table, and performed basic exploratory analysis such as filtering, normalization, grouping, and visualization. For further analysis, refer to the Croissant schema for detailed field descriptions and possible downstream biological or clinical applications.